# Volumetric Modeling (Voxel Grids & TSDF)

This notebook shows how to represent 3D objects using **voxels** (3D pixels) and how to combine multiple scans using **TSDF fusion**.

**Goals:**
- What are voxel grids and why use them
- How to convert meshes to voxels
- Basics of TSDF fusion for combining scans
- When to use voxels vs meshes

## 1. What are Voxel Grids?

Think of voxels like **3D pixels** - they divide space into small cubes. Each cube is either:
- **Occupied** (part of the object) 
- **Empty** (just air)

**Why use voxels?**
- Easy to check if a point is inside an object
- Good for collision detection in robots
- Can calculate volumes easily
- Regular grid structure is simple to work with

In [3]:
import open3d as o3d
import numpy as np

print("Setting up 3D objects...")

# Load the Stanford Bunny (our main example)
dataset = o3d.data.BunnyMesh()
mesh = o3d.io.read_triangle_mesh(dataset.path)
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.7, 0.3, 0.3])  # Red color

print(f"Loaded bunny mesh with {len(mesh.vertices)} vertices")
print("Ready to create voxels!")

Setting up 3D objects...
Loaded bunny mesh with 35947 vertices
Ready to create voxels!


## 2. Creating Voxel Grids from Meshes

In [4]:
# First, let's see the original mesh
print("Original mesh:")
o3d.visualization.draw_geometries([mesh], window_name="Original Bunny Mesh")

# Now convert to voxels
voxel_size = 0.02  # 2cm cubes
voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh(mesh, voxel_size=voxel_size)

print(f"Created {len(voxel_grid.get_voxels())} voxels")
print(f"Each voxel is {voxel_size}m = {voxel_size*100}cm on each side")

# Show the voxel version
print("Voxel version:")
o3d.visualization.draw_geometries([voxel_grid], window_name="Bunny as Voxels")

Original mesh:
Created 206 voxels
Each voxel is 0.02m = 2.0cm on each side
Voxel version:


## 3. Voxels from Point Clouds

voxels can also be created from point clouds (just scattered 3D points).

In [5]:
# Create a point cloud from the mesh
pcd = mesh.sample_points_poisson_disk(2000)  # 2000 random points on surface
pcd.paint_uniform_color([0.3, 0.7, 0.3])  # Green color

print(f"Created point cloud with {len(pcd.points)} points")

# Show the point cloud
print("Point cloud:")
o3d.visualization.draw_geometries([pcd], window_name="Point Cloud")

# Convert point cloud to voxels
voxel_grid_pcd = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=voxel_size)

print(f"Point cloud created {len(voxel_grid_pcd.get_voxels())} voxels")

# Show voxels from point cloud
print("Voxels from point cloud:")
o3d.visualization.draw_geometries([voxel_grid_pcd], window_name="Voxels from Points")

Created point cloud with 2000 points
Point cloud:
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
Point cloud created 186 voxels
Voxels from point cloud:


## 4. What is TSDF Fusion?

**TSDF** = Truncated Signed Distance Function

**The Problem:** When you scan an object with a 3D camera, you only see one side. To get the complete object, you need multiple scans from different angles.

**TSDF Solution:**
1. For each voxel, store how far it is from the nearest surface
2. **Positive distance** = outside the object
3. **Negative distance** = inside the object  
4. When you get a new scan, **average** the distances to reduce noise
5. Extract the final surface where distance = 0

**Real example:** RIIICO's robots walk around a machine part, taking depth photos from different angles. TSDF fusion combines all these photos into one complete 3D model.

In [11]:
# Create a TSDF volume (empty at first)
voxel_length = 0.004  # 4mm voxels (smaller = more detail)
sdf_trunc = 0.02      # Only store distances within 2cm of surface

tsdf_volume = o3d.pipelines.integration.ScalableTSDFVolume(
    voxel_length=voxel_length,
    sdf_trunc=sdf_trunc,
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
)

print(f"Created TSDF volume with {voxel_length*1000}mm voxels")
print(f"Will store distances up to {sdf_trunc*100}cm from surfaces")
print("Ready to integrate RGB-D scans!")

Created TSDF volume with 4.0mm voxels
Will store distances up to 2.0cm from surfaces
Ready to integrate RGB-D scans!


## 5. TSDF Demo with Real Data

Let's try TSDF with real RGB-D (color + depth) images.

In [12]:
try:
    # Load real RGB-D dataset
    print("Loading RGB-D images...")
    dataset = o3d.data.SampleRedwoodRGBDImages()
    
    # Camera settings (these match the dataset)
    intrinsics = o3d.camera.PinholeCameraIntrinsic()
    intrinsics.set_intrinsics(640, 480, 525.0, 525.0, 319.5, 239.5)
    
    # Create new TSDF volume for this demo
    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=0.004,
        sdf_trunc=0.02,
        color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
    )
    
    # Process first few images (to keep it simple)
    n_images = min(5, len(dataset.color_paths))
    print(f"Processing {n_images} RGB-D images...")
    
    for i in range(n_images):
        # Load color and depth images
        color = o3d.io.read_image(dataset.color_paths[i])
        depth = o3d.io.read_image(dataset.depth_paths[i])
        
        # Combine into RGB-D image
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            color, depth,
            depth_scale=1000.0,  # Depth images are in mm
            depth_trunc=3.0,     # Ignore points beyond 3m
            convert_rgb_to_intensity=False
        )
        
        # Add to TSDF volume (assuming camera doesn't move for simplicity)
        volume.integrate(rgbd, intrinsics, np.identity(4))
        print(f"  Integrated image {i+1}/{n_images}")
    
    # Extract the final mesh
    print("Extracting final mesh from TSDF...")
    tsdf_mesh = volume.extract_triangle_mesh()
    tsdf_mesh.compute_vertex_normals()
    
    if len(tsdf_mesh.vertices) > 0:
        print(f"Success! Created mesh with {len(tsdf_mesh.vertices)} vertices")
        tsdf_mesh.paint_uniform_color([0.3, 0.3, 0.7])  # Blue color
        o3d.visualization.draw_geometries([tsdf_mesh], window_name="TSDF Reconstruction")
    else:
        print("No mesh generated - this can happen with limited data")
        
except Exception as e:
    print(f"TSDF demo had an issue: {e}")
    print("This is normal in some environments. The concept is what matters!")

Loading RGB-D images...
Processing 5 RGB-D images...
  Integrated image 1/5
  Integrated image 2/5
  Integrated image 3/5
  Integrated image 4/5
  Integrated image 5/5
Extracting final mesh from TSDF...
Success! Created mesh with 1058133 vertices
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 


## 6. Voxels vs Meshes - Which to Use?

Let's compare our different representations:

In [13]:
print("=== COMPARISON: Mesh vs Voxels ===")
print()

# Mesh stats
print(f"ORIGINAL MESH:")
print(f"  Vertices: {len(mesh.vertices)}")
print(f"  Triangles: {len(mesh.triangles)}")
print(f"  Good for: Smooth surfaces, rendering, animation")
print(f"  Bad for: Collision detection, volume calculations")
print()

# Voxel stats  
print(f"VOXEL GRID:")
print(f"  Voxels: {len(voxel_grid.get_voxels())}")
print(f"  Size: {voxel_size}m per voxel")
print(f"  Good for: Spatial queries, collision detection, volumes")
print(f"  Bad for: Looks blocky, uses more memory")
print()

# Show both together
print("Showing both together (red mesh + colored voxels):")
o3d.visualization.draw_geometries([mesh, voxel_grid], 
                                  window_name="Mesh + Voxels Comparison")

=== COMPARISON: Mesh vs Voxels ===

ORIGINAL MESH:
  Vertices: 35947
  Triangles: 69451
  Good for: Smooth surfaces, rendering, animation
  Bad for: Collision detection, volume calculations

VOXEL GRID:
  Voxels: 206
  Size: 0.02m per voxel
  Good for: Spatial queries, collision detection, volumes
  Bad for: Looks blocky, uses more memory

Showing both together (red mesh + colored voxels):
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 


## 7. When to Use What?

**Use Voxel Grids when:**
- Fast collision detection (robots avoiding obstacles)
- Volume calculations (how much material in a part?)
- Spatial queries (what's at this location?)
- Regular grid operations (filtering, morphology)

**Use TSDF Fusion when:**
- Multiple RGB-D scans of the same object
- Noisy sensor data that needs averaging
- Need to build complete 3D models from partial views
- Robot mapping applications

**Use Meshes when:**
- Smooth, pretty surfaces
- Efficient rendering
- Animation and deformation
- Compact file sizes

## 8. Key Takeaways

🎯 **What I learned:**

1. **Voxels are 3D pixels** - they divide space into regular cubes
2. **Easy conversion** from meshes and point clouds to voxels
3. **TSDF fusion combines multiple scans** into one complete model
4. **Trade-offs exist**: voxels are blocky but good for spatial operations